### Data Engineering Assistant

#### Building a dense vector retrieval application using **LangChain**, **Gemini Embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load the Big Book of Data Engineering PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

In [2]:
#import warnings
#warnings.filterwarnings("ignore")

### 🛠️ Environmental Setup

In this step, we install `uv`, create a virtual environment, prepare a `requirements.txt` file, and install the Python packages needed for the notebook.

### Step 1: Install `uv`
Use the official installer to install `uv` on your system.

**Windows:**
```bash
powershell -ExecutionPolicy ByPass -c "irm [https://astral.sh/uv/install.ps1](https://astral.sh/uv/install.ps1) | iex"

**macOS and Linux:**
```bash
curl -LsSf [https://astral.sh/uv/install.sh](https://astral.sh/uv/install.sh) | sh
```

### Step 2: Check the installation
Open a new terminal and verify that `uv` is available.
```bash
uv --version
```
### Initialize a new project (this creates a pyproject.toml and standard structure)
```bash
uv init
```
### Step 3: Create a virtual environment
Create and activate the project environment before installing dependencies.
```bash
uv venv
.venv/bin/activate
```

### Step 4: Add the dependencies
Create a `requirements.txt` file with the packages required for this project.
```bash
langchain-core
langchain-community
langchain-text-splitters
langchain-google-genai
langchain-qdrant
qdrant-client
python-dotenv
ipykernel
jupyter
pypdf
```

### Step 5: Install the packages
Use `uv` to install everything from `requirements.txt`.
```bash
uv pip install -r requirements.txt
```

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_google_genai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\zeesh\AppData\Local\Temp\ipykernel_25512\3140196580.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


An embedding model turns a piece of text into a numeric vector. If two pieces of text are related, their vectors should be close to each other. We measure that closeness with cosine similarity, which tells us how aligned two vectors are.

Why this matters
In RAG, we embed each document chunk once, then embed the user’s question and search for the most similar chunks. That retrieval step is what helps the model answer from the right part of the PDF.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")